# 校园快递包裹分类识别 —— 自动定位项目目录 + 一键训练 Notebook

**使用方式（VSCode + Colab GPU 插件 / 网页版 Colab 都适用）：**

1. 直接打开这个 `.ipynb`，连接 Colab GPU 内核
2. 运行第 0 个 cell：
   - 如果这台 Colab 运行时**已经有**项目代码（比如挂载了 Drive，或者你之前上传过），会自动找到并 `cd` 过去
   - 如果**没有**（比如你就是单独打开这一个 ipynb），会自动弹出上传框，让你传一次 `campus-express-yolov8.zip`，
     解压后自动定位过去，后面所有 cell 都不用再管路径
3. 从上到下依次运行剩下的 cell

**注意：** 上传是"每次开新的 Colab 运行时都要传一次"（Colab 运行时重启后 `/content` 会清空）。
如果不想每次都传，把项目文件夹放进 Google Drive，然后在第0个 cell 前先加一个
`from google.colab import drive; drive.mount('/content/drive')`，之后自动定位就能直接找到 Drive 里那份，不用反复上传。


## 0. 自动定位项目根目录

In [ ]:
import os
import subprocess
import zipfile

MARKERS = ("requirements.txt", "data/scripts", "training/train.py")

def find_project_root(start, markers=MARKERS):
    cur = os.path.abspath(start)
    while True:
        if all(os.path.exists(os.path.join(cur, m)) for m in markers):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            return None
        cur = parent

def scan_common_dirs(markers=MARKERS):
    """兜底：在常见根目录下用 find 搜 training/train.py 反推项目根目录"""
    for base in ("/content", "/kaggle/working", os.path.expanduser("~"), "/mnt", "/workspace"):
        if not os.path.exists(base):
            continue
        try:
            out = subprocess.run(
                ["find", base, "-maxdepth", "6", "-path", "*/training/train.py"],
                capture_output=True, text=True, timeout=30
            ).stdout.strip().splitlines()
        except Exception:
            continue
        for hit in out:
            candidate = os.path.dirname(os.path.dirname(hit))
            if os.path.exists(os.path.join(candidate, "requirements.txt")):
                return candidate
    return None

IN_COLAB = 'google.colab' in str(get_ipython())

PROJECT_ROOT = find_project_root(os.getcwd()) or scan_common_dirs()

if PROJECT_ROOT is None:
    print("❌ 没在当前文件系统里找到项目代码。")
    if IN_COLAB:
        print("这台 Colab 运行时还没有项目文件，现在弹出上传框，请选择 campus-express-yolov8.zip：")
        from google.colab import files
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]

        extract_dir = "/content/work"
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(zip_name, "r") as zf:
            zf.extractall(extract_dir)

        PROJECT_ROOT = find_project_root(extract_dir) or scan_common_dirs()
        if PROJECT_ROOT is None:
            raise SystemExit(
                "解压后仍未找到项目标志文件，请检查上传的 zip 是否就是 campus-express-yolov8 项目本身"
                "（zip 解压后应该能看到 requirements.txt / data/scripts / training/train.py）。"
            )
    else:
        raise SystemExit(
            "当前不是 Colab 网页环境（可能是本地内核/远程内核但没有上传入口），"
            "请手动把 PROJECT_ROOT 改成项目在当前文件系统里的绝对路径，例如：\n"
            "  PROJECT_ROOT = '/home/xxx/campus-express-yolov8'\n"
            "改完后重新运行本 cell（把上面几行 raise SystemExit 那段临时注释掉即可）。"
        )

os.chdir(PROJECT_ROOT)
print("✅ 已定位并切换到项目根目录:", PROJECT_ROOT)
!ls


## 1. 检查 GPU

In [ ]:
!nvidia-smi


## 2. 安装依赖

In [ ]:
!pip install -q -r requirements.txt


## 3. 检查数据是否已就位

只是打印检查，不会自动帮你下载数据 —— 公开数据集/校园数据集需要你提前放到
`data/public_raw/` 和 `data/campus_raw/`（参考项目 README）。


In [ ]:
def check(path, desc):
    ok = os.path.exists(path)
    mark = "✅" if ok else "❌"
    print(f"{mark} {desc}: {path}")
    return ok

has_public = check("data/public_raw", "公开数据集原始目录")
has_campus_img = check("data/campus_raw/images", "校园数据集图像目录")
has_campus_lbl = check("data/campus_raw/labels", "校园数据集标注目录")

if not has_public:
    print("\n⚠️ 未检测到 data/public_raw，请先把 Roboflow 下载解压的数据集放进去，"
          "或者运行 data/scripts/download_public_dataset.py 用 API 下载。")


## 4. 整理公开数据集为统一 YOLO 格式

In [ ]:
!python data/scripts/prepare_public_dataset.py --src data/public_raw --out data/public_yolo


## 5. 配置类别映射（必须核对，运行前先看第4步打印出的类别名）

直接改这个 cell 里的 `PUBLIC_CLASS_MAP`，运行后会自动写回 `data/scripts/merge_datasets.py`。


In [ ]:
import re

FINAL_CLASSES = ["纸箱", "文件袋", "塑料袋", "泡沫箱"]

PUBLIC_CLASS_MAP = {
    "boxes": "纸箱",
    "parcel": "纸箱",
    "good-parcel": "纸箱",
    "package": "纸箱",
    "label": None,
}

classes_repr = "[" + ", ".join(f'"{c}"' for c in FINAL_CLASSES) + "]"
map_lines = "\n".join(
    f'    "{k}": {repr(v) if v is None else chr(34)+v+chr(34)},' for k, v in PUBLIC_CLASS_MAP.items()
)
patch = f'FINAL_CLASSES = {classes_repr}\n\nPUBLIC_CLASS_MAP = {{\n{map_lines}\n}}'

with open("data/scripts/merge_datasets.py", "r", encoding="utf-8") as f:
    content = f.read()

pattern = re.compile(r'FINAL_CLASSES = \[.*?\n\}', re.S)
new_content, n = pattern.subn(patch, content, count=1)
assert n == 1, "没匹配到原有的 FINAL_CLASSES/PUBLIC_CLASS_MAP 段落，请检查 merge_datasets.py 是否被手动改动过"

with open("data/scripts/merge_datasets.py", "w", encoding="utf-8") as f:
    f.write(new_content)

print("✅ 类别映射已写入 data/scripts/merge_datasets.py")
!sed -n '1,30p' data/scripts/merge_datasets.py


## 6. 合并 + 清洗 + 划分 + 增强

In [ ]:
!python data/scripts/merge_datasets.py
!python data/scripts/clean_dataset.py
!python data/scripts/split_dataset.py
!python data/scripts/augment.py


## 7. 阶段1训练：公开数据集预训练（COCO权重起步）

In [ ]:
!python training/train.py --stage 1 --config training/configs/stage1_public_pretrain.yaml


## 8. 阶段2训练：校园数据集迁移微调（用阶段1权重起步）

In [ ]:
!python training/train.py --stage 2 --config training/configs/stage2_campus_finetune.yaml \
    --weights training/runs/stage1/weights/best.pt


## 9. 评估（生成论文第六章可用的指标表）

In [ ]:
!python training/eval.py --weights training/runs/stage2/weights/best.pt --data data/dataset/data.yaml
import pandas as pd
pd.read_csv("training/eval_report.csv")


## 10.（可选）启动推理服务本地测试

需要先把 `training/runs/stage2/weights/best.pt` 拷贝到 `system/backend/models/best.pt`。


In [ ]:
!cp training/runs/stage2/weights/best.pt system/backend/models/best.pt
import subprocess, time
proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="system/backend"
)
time.sleep(3)
print("推理服务已启动: http://127.0.0.1:8000/detect  (Colab 需要用 ngrok/colab端口转发才能外部访问)")
print("停止服务请运行: proc.terminate()")
